# Time Series Forecasting with ARIMA Models In Python [Part 2]

Run all cells in order in a fresh Python 3.11 kernel. Complete original datasets are pinned and SHA-256 verified. All original gist examples are preserved verbatim in nonexecuting appendices. Code and matching explanations were corrected with user approval. Stock examples are historical educational exercises, not investment advice. Same-day cross-company regression is distinguished from the past-only lag forecast. No keys or GPU. All writes use an owned temporary workspace.

In [ ]:
import os,sys,subprocess,hashlib,urllib.request,urllib.parse,warnings,random
from pathlib import Path
from tempfile import TemporaryDirectory
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','pandas==3.1.0','numpy==2.3.5','scipy==1.16.3','matplotlib==3.10.8','statsmodels==0.14.6','pmdarima==2.1.1','scikit-learn==1.9.1','joblib==1.6.0','certifi==2026.10.23'],check=True)
import certifi
os.environ['SSL_CERT_FILE']=certifi.where()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
np.random.seed(42);random.seed(42)
pd.set_option('display.max_rows',12);pd.set_option('display.max_columns',12)
plt.rcParams.update({'figure.dpi':80,'figure.figsize':(10,5),'font.size':10})
workspace=TemporaryDirectory(prefix='tdb-time-series-11-');work=Path(workspace.name);previous_cwd=Path.cwd()
source_revision='65d4ae5d3436be3c0a921a3dcbda1b52e3ac8597'
source_files={'Arima Models in Python/candy_production.csv': '6b09ba45d482a23459499b98006d2e4911f31a06ded14ac2a1192dd5279c9056', 'Arima Models in Python/amazon_close.csv': 'f73498d0e1efa5cf9936bd4d2f810259c67ef39b3436f406b194de778fdf40ec', 'Arima Models in Python/co2.csv': 'a777e6a6000ff2dbf9bc12ff652a73a59b3d9c9e459f4330c3fd2574c6768b5c', 'Arima Models in Python/earthquakes.csv': '0d6e92b04f0337d13ea6055785032189763652ee4ec4bffbd80961964d6bbff9', 'Machine Learning For Time Series/prices.csv': 'd6fbdd4831ec8fd5884a690740d38bfc9dff44cfde0decc78ba122e70f58ebf6', 'Machine Learning For Time Series/preprocessed_prices.csv': '23e45fb2fbba7cc93ede87805546df86cba5531e82cc8e3cbdb4d63ddab3b176'}
for relative,digest in source_files.items():
    url='https://raw.githubusercontent.com/youssefHosni/Time-Series-Analysis/'+source_revision+'/'+urllib.parse.quote(relative)
    with urllib.request.urlopen(url,timeout=120)as response:payload=response.read()
    assert hashlib.sha256(payload).hexdigest()==digest,relative
    (work/Path(relative).name).write_bytes(payload)
os.chdir(work)
def checked_fit(model):
    result=model.fit(disp=False,maxiter=1000)
    if not result.mle_retvals.get('converged',False):
        result=model.fit(start_params=result.params,method='powell',disp=False,maxiter=2000)
    assert result.mle_retvals.get('converged',False),'Model did not converge'
    assert np.isfinite(result.params).all() and np.isfinite(result.aic)
    return result
print('Python:',sys.version.split()[0],'| pandas:',pd.__version__,'| NumPy:',np.__version__)
print('Six complete original datasets verified at pinned revision. No API keys or GPU.')

In [ ]:
earthquake = pd.read_csv('earthquakes.csv',index_col='date',parse_dates=True).sort_index().asfreq('YS')
earthquake_series = earthquake['earthquakes_per_year']
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# Make ACF plot
plot_acf(earthquake['earthquakes_per_year'], lags=10, zero=False)
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 1 completed.')

In [ ]:
# Make PACF plot
plot_pacf(earthquake['earthquakes_per_year'], lags=10, zero=False)
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 2 completed.')

In [ ]:
grid_fits = {}; grid_failures = []; order_aic_bic = []
for p in range(3):
    for q in range(3):
        try:
            fitted = checked_fit(SARIMAX(earthquake_series,order=(p,0,q),trend='c'))
            grid_fits[(p,q)] = fitted
            order_aic_bic.append((p,q,fitted.aic,fitted.bic))
        except (ValueError,np.linalg.LinAlgError,AssertionError) as exc:
            grid_failures.append((p,q,str(exc)))
order_df=pd.DataFrame(order_aic_bic,columns=['p','q','aic','bic'])
assert not order_df.empty
print('Models sorted by AIC');print(order_df.sort_values('aic').to_string(index=False))
print('Models sorted by BIC');print(order_df.sort_values('bic').to_string(index=False))
print('Failed candidates:',grid_failures)

earthquake_fits = grid_fits.copy()
earthquake_orders = order_df.copy()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 3 completed.')

In [ ]:
best_earthquake_order = tuple(order_df.sort_values('aic').iloc[0][['p','q']].astype(int))
results = earthquake_fits[best_earthquake_order]
earthquake_result = results
residuals=results.resid
print('Selected earthquake (p,q):',best_earthquake_order)
print(residuals)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 4 completed.')

In [ ]:
# The mean absolute error
mae = np.mean(np.abs(residuals))
print(mae)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 5 completed.')

In [ ]:
# Create the 4 diagostics plots
results.plot_diagnostics(figsize=(10,10))
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 6 completed.')

In [ ]:
# Summary statistics
print(results.summary())
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 7 completed.')

In [ ]:
co2 = pd.read_csv('co2.csv',
                     index_col='date',
                     parse_dates=True).sort_index().asfreq('MS')
co2.plot()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 8 completed.')

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Run Dicky-Fuller test
result = adfuller(co2.iloc[:,0])

# Print test statistic
print('The test stastics:', result[0])

# Print p-value
print("The p-value:",result[1])
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 9 completed.')

In [ ]:
co2_diff = co2.diff()
co2_diff = co2_diff.dropna()
co2_diff.plot()

# Run Dicky-Fuller test
result_diff = adfuller(co2_diff.iloc[:,0])

# Print test statistic
print('The test stastics:', result_diff[0])

# Print p-value
print("The p-value:",result_diff[1])
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 10 completed.')

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Create figure
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(12,8))
 
# Plot the ACF of savings on ax1
plot_acf(co2_diff, lags=20, zero=False, ax=ax1)

# Plot the PACF of savings on ax2
plot_pacf(co2_diff, lags=20, zero=False, ax=ax2)

plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 11 completed.')

In [ ]:
grid_fits = {}; grid_failures = []; order_aic_bic = []
for p in range(5):
    for q in range(5):
        try:
            fitted = checked_fit(SARIMAX(co2['CO2_ppm'],order=(p,1,q),trend='c'))
            grid_fits[(p,q)] = fitted
            order_aic_bic.append((p,q,fitted.aic,fitted.bic))
        except (ValueError,np.linalg.LinAlgError,AssertionError) as exc:
            grid_failures.append((p,q,str(exc)))
order_df=pd.DataFrame(order_aic_bic,columns=['p','q','aic','bic'])
assert not order_df.empty
print('Models sorted by AIC');print(order_df.sort_values('aic').to_string(index=False))
print('Models sorted by BIC');print(order_df.sort_values('bic').to_string(index=False))
print('Failed candidates:',grid_failures)

co2_fits = grid_fits.copy()
co2_orders = order_df.copy()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 12 completed.')

In [ ]:
best_co2_order=tuple(order_df.sort_values('aic').iloc[0][['p','q']].astype(int))
results=co2_fits[best_co2_order]
co2_result=results
print('Selected CO2 (p,1,q):',(best_co2_order[0],1,best_co2_order[1]))
results.plot_diagnostics(figsize=(10,8));plt.show()
print(results.summary())
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 13 completed.')

In [ ]:
candy=pd.read_csv('candy_production.csv',index_col='date',parse_dates=True).sort_index().asfreq('MS')
candy_raw=candy['IPG3113N'].copy()
# Import
from statsmodels.tsa.seasonal import seasonal_decompose
# Decompose data
decomp_results = seasonal_decompose(candy, period=12)

# Plot decomposed data
plt.rcParams["figure.figsize"] = (10,15)
decomp_results.plot()
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 14 completed.')

In [ ]:
candy_detrended=candy_raw-candy_raw.rolling(5).mean()
fig,ax=plt.subplots(figsize=(10,4))
plot_acf(candy_detrended.dropna(),ax=ax,lags=25,zero=False);plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 15 completed.')

In [ ]:
p,d,q=0,1,3
P,D,Q,S=3,1,0,12
model=SARIMAX(candy_raw,order=(p,d,q),seasonal_order=(P,D,Q,S))
print('Non-seasonal order:',model.order,'seasonal order:',model.seasonal_order)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 16 completed.')

In [ ]:
S=12
candy_seasonal_diff=candy_raw.diff(S).dropna()
candy_seasonal_diff.plot(title='Twelve-month seasonal difference');plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 17 completed.')

In [ ]:
candy_diff=candy_raw.diff(12).diff().dropna()
candy_diff.plot(title='Seasonal difference followed by one-step difference');plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 18 completed.')

In [ ]:
# Run Dicky-Fuller test
result = adfuller(candy_diff)

# Print test statistic
print('The test stastics:', result[0])

# Print p-value
print("The p-value:",result[1])

for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 19 completed.')

In [ ]:
# find the non-seasonal model prameters 

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Create figure
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(12,8))
 
# Plot the ACF of savings on ax1
plot_acf(candy_diff, lags=20, zero=False, ax=ax1)

# Plot the PACF of savings on ax2
plot_pacf(candy_diff, lags=20, zero=False, ax=ax2)

plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 20 completed.')

In [ ]:
grid_fits = {}; grid_failures = []; order_aic_bic = []
for p in range(5):
    for q in range(5):
        try:
            fitted = checked_fit(SARIMAX(candy_diff,order=(p,0,q),trend='c'))
            grid_fits[(p,q)] = fitted
            order_aic_bic.append((p,q,fitted.aic,fitted.bic))
        except (ValueError,np.linalg.LinAlgError,AssertionError) as exc:
            grid_failures.append((p,q,str(exc)))
order_df=pd.DataFrame(order_aic_bic,columns=['p','q','aic','bic'])
assert not order_df.empty
print('Models sorted by AIC');print(order_df.sort_values('aic').to_string(index=False))
print('Models sorted by BIC');print(order_df.sort_values('bic').to_string(index=False))
print('Failed candidates:',grid_failures)

candy_fits=grid_fits.copy()
candy_orders=order_df.copy()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 21 completed.')

In [ ]:
# Plotting seasonal ACF and PACF

# Create figure
fig, (ax1, ax2) = plt.subplots(2,1)

# Plot seasonal ACF
plot_acf(candy_diff, lags=[12,24,36,48,60,72, 84, 96 ], ax=ax1)

# Plot seasonal PACF
plot_pacf(candy_diff, lags=[12,24,36,48,60,72, 84, 96], ax=ax2)
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 22 completed.')

In [ ]:
best_candy_order=tuple(candy_orders.sort_values('aic').iloc[0][['p','q']].astype(int))
p,q=best_candy_order
P,D,Q,S=3,1,0,12
model=SARIMAX(candy_raw,order=(p,1,q),seasonal_order=(P,D,Q,S))
results=checked_fit(model)
manual_seasonal_result=results
print('Selected nonseasonal order:',(p,1,q),'seasonal order:',(P,D,Q,S))
print(results.summary())
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 23 completed.')

In [ ]:
import pmdarima as pm
nonseasonal_candidates=pm.auto_arima(candy_raw,seasonal=False,maxiter=300,return_valid_fits=True,suppress_warnings=False,error_action='trace',stepwise=True)
nonseasonal_candidates=[v for v in nonseasonal_candidates if v.arima_res_.mle_retvals.get('converged',False)]
assert nonseasonal_candidates
results=min(nonseasonal_candidates,key=lambda v:v.aic())
nonseasonal_auto=results
print(results);print(results.summary());results.plot_diagnostics(figsize=(10,8));plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 24 completed.')

In [ ]:
seasonal_candidates=pm.auto_arima(candy_raw,seasonal=True,m=12,D=1,start_P=1,start_Q=1,max_P=4,max_Q=4,information_criterion='aic',trace=True,error_action='trace',stepwise=True,maxiter=300,return_valid_fits=True,suppress_warnings=False)
seasonal_candidates=[v for v in seasonal_candidates if v.arima_res_.mle_retvals.get('converged',False)]
assert seasonal_candidates
results=min(seasonal_candidates,key=lambda v:v.aic())
seasonal_auto=results
print(results);print(results.summary());results.plot_diagnostics(figsize=(10,8));plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 25 completed.')

In [ ]:
#Saving model objects
# Import joblib
import joblib
# Select a filepath
filepath = 'model.pkl'
# Save model to filepath
joblib.dump(results, filepath)

# Load the model 
# Select a filepath
filepath ='model.pkl'
# Load model object from filepath
loaded_model = joblib.load(filepath)
np.testing.assert_allclose(loaded_model.predict(24),results.predict(24),rtol=0,atol=0)
print('Saved/loaded 24-step forecasts match exactly.')
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 26 completed.')

In [ ]:
from numpy.testing import assert_allclose
from scipy.stats import norm
def independent_forecast(result,steps):
    ssm=result.model.ssm
    Z=ssm.design[:,:,0];T=ssm.transition[:,:,0];R=ssm.selection[:,:,0];Q=ssm.state_cov[:,:,0];H=ssm.obs_cov[:,:,0]
    obs=ssm.obs_intercept[:,0];drift=ssm.state_intercept[:,0]
    state=result.filter_results.predicted_state[:,-1].copy()
    covariance=result.filter_results.predicted_state_cov[:,:,-1].copy()
    means=[];variances=[]
    for _ in range(steps):
        means.append(float((Z@state+obs)[0]));variances.append(float((Z@covariance@Z.T+H)[0,0]))
        state=T@state+drift;covariance=T@covariance@T.T+R@Q@R.T
    means=np.asarray(means);variances=np.asarray(variances)
    intervals=np.column_stack([means-norm.ppf(.975)*np.sqrt(variances),means+norm.ppf(.975)*np.sqrt(variances)])
    return means,intervals
raw=candy_raw.to_numpy()
expected_diff=raw[13:]-raw[1:-12]-raw[12:-1]+raw[:-13]
assert len(candy_raw)==564 and len(candy_diff)==551
assert_allclose(candy_diff,expected_diff,atol=1e-12)
assert_allclose(candy_seasonal_diff,raw[12:]-raw[:-12],atol=1e-12)
assert_allclose(candy_detrended.iloc[4:],raw[4:]-np.convolve(raw,np.ones(5)/5,mode='valid'),atol=1e-12)
assert len(earthquake_series)==99 and len(co2)==727
for fits,table,count in [(earthquake_fits,earthquake_orders,9),(co2_fits,co2_orders,25),(candy_fits,candy_orders,25)]:
    assert len(fits)==len(table)==count
    for (p,q),fit in fits.items():
        assert fit.mle_retvals['converged']
        row=table.loc[(table.p==p)&(table.q==q)].iloc[0]
        assert_allclose(row.aic,-2*fit.llf+2*fit.df_model,rtol=1e-12)
        assert_allclose(row.bic,-2*fit.llf+np.log(fit.nobs_effective)*fit.df_model,rtol=1e-12)
    selected=tuple(table.sort_values('aic').iloc[0][['p','q']].astype(int))
    assert fits[selected].aic==min(v.aic for v in fits.values())
assert_allclose(earthquake_result.resid,earthquake_series.to_numpy()-earthquake_result.fittedvalues.to_numpy(),atol=1e-12)
assert_allclose(mae,np.mean(np.abs(earthquake_result.resid)),atol=1e-12)
assert manual_seasonal_result.model.order==(best_candy_order[0],1,best_candy_order[1])
assert manual_seasonal_result.model.seasonal_order==(3,1,0,12)
assert seasonal_auto.arima_res_.mle_retvals['converged']
means,intervals=independent_forecast(loaded_model.arima_res_,24)
auto_means,auto_intervals=loaded_model.predict(24,return_conf_int=True)
assert_allclose(auto_means,means,atol=1e-8)
assert_allclose(auto_intervals,intervals,atol=1e-8)
assert_allclose(loaded_model.predict(24),seasonal_auto.predict(24),rtol=0,atol=0)
metrics={'earthquake_rows':99,'co2_rows':727,'candy_rows':564,'combined_difference_rows':551,'candidate_counts':[len(earthquake_fits),len(co2_fits),len(candy_fits)],'earthquake_order':list(best_earthquake_order),'co2_order':[best_co2_order[0],1,best_co2_order[1]],'candy_nonseasonal_order':list(best_candy_order),'nonseasonal_auto_order':list(nonseasonal_auto.order),'seasonal_auto_order':list(seasonal_auto.order),'seasonal_auto_seasonal_order':list(seasonal_auto.seasonal_order),'seasonal_forecast_24_last':float(np.asarray(auto_means)[-1]),'seasonal_forecast_24_interval':np.asarray(auto_intervals)[-1].tolist()}
print('All 59 candidate fits, independent differencing/AIC/BIC/residuals, full seasonal forecast state/covariance and save/reload checks passed.')
print('METRICS:',metrics)

os.chdir(previous_cwd)
workspace.cleanup()
print('Owned temporary workspace cleaned up.')

## Original source code

### Original block 1

```text
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# Make ACF plot
plot_acf(earthquake['earthquakes_per_year'], lags=10, zero=False)
plt.show()
```

### Original block 2

```text
# Make PACF plot
plot_pacf(earthquake['earthquakes_per_year'], lags=10, zero=False)
plt.show()
```

### Original block 3

```text
import statsmodels.api as sm
order_aic_bic =[]

# Loop over p values from 0-2
for p in range(3):
    # Loop over q values from 0-2
    for q in range(3):
      
        try:
            # create and fit ARMA(p,q) model
            model = sm.tsa.statespace.SARIMAX(earthquake['earthquakes_per_year'], order=(p, 0, q))
            results = model.fit()
            
            # Print order and results
            order_aic_bic.append((p, q, results.aic, results.bic))            
        except:
            print(p, q, None, None)
            
# Make DataFrame of model order and AIC/BIC scores
order_df = pd.DataFrame(order_aic_bic, columns=['p', 'q', 'aic','bic'])

# lets sort them by AIC and BIC

# Sort by AIC
print("Sorted by AIC ")
print("\n")
print(order_df.sort_values('aic').reset_index(drop=True))

# Sort by BIC
print("Sorted by BIC ")
print("\n")
print(order_df.sort_values('bic').reset_index(drop=True))
```

### Original block 4

```text
# The model with the best p and q found from pervious step
model = sm.tsa.statespace.SARIMAX(earthquake['earthquakes_per_year'], order=(1, 0, 1))
# Fit model
results = model.fit()
# Assign residuals to variable
residuals = results.resid
print(residuals)
```

### Original block 5

```text
# The mean absolute error
mae = np.mean(np.abs(residuals))
print(mae)
```

### Original block 6

```text
# Create the 4 diagostics plots
results.plot_diagnostics(figsize=(10,10))
plt.show()
```

### Original block 7

```text
# Summary statistics
print(results.summary())
```

### Original block 8

```text
co2 = pd.read_csv('co2.csv',
                     index_col='date',
                     parse_dates=True)
co2.plot()
```

### Original block 9

```text
from statsmodels.tsa.stattools import adfuller

# Run Dicky-Fuller test
result = adfuller(co2)

# Print test statistic
print('The test stastics:', result[0])

# Print p-value
print("The p-value:",result[1])
```

### Original block 10

```text
co2_diff = co2.diff()
co2_diff = co2_diff.dropna()
co2_diff.plot()

# Run Dicky-Fuller test
result_diff = adfuller(co2_diff)

# Print test statistic
print('The test stastics:', result_diff[0])

# Print p-value
print("The p-value:",result_diff[1])
```

### Original block 11

```text
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Create figure
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(12,8))
 
# Plot the ACF of savings on ax1
plot_acf(co2_diff, lags=20, zero=False, ax=ax1)

# Plot the PACF of savings on ax2
plot_pacf(co2_diff, lags=20, zero=False, ax=ax2)

plt.show()
```

### Original block 12

```text
import statsmodels.api as sm
order_aic_bic =[]

# Loop over p values from 0-4
for p in range(5):
    # Loop over q values from 0-4
    for q in range(5):
      
        try:
            # create and fit ARMA(p,q) model
            model = sm.tsa.statespace.SARIMAX(co2, order=(p, 1, q))
            results = model.fit()
            
            # Print order and results
            order_aic_bic.append((p, q, results.aic, results.bic))            
        except:
            print(p, q, None, None)
            
# Make DataFrame of model order and AIC/BIC scores
order_df = pd.DataFrame(order_aic_bic, columns=['p', 'q', 'aic','bic'])

# lets sort them by AIC and BIC

# Sort by AIC
print("Models sorted by AIC ")
print("\n")
print(order_df.sort_values('aic').reset_index(drop=True))

# Sort by BIC
print("Models sorted by BIC ")
print("\n")
print(order_df.sort_values('bic').reset_index(drop=True))
```

### Original block 13

```text
# Create and fit model
model = sm.tsa.statespace.SARIMAX(co2, order=(4,1,4), trend='c')
results = model.fit()

# Create the 4 diagostics plots
results.plot_diagnostics(figsize=(10,10))
plt.show()

# Print summary
print(results.summary())
```

### Original block 14

```text
# Import
from statsmodels.tsa.seasonal import seasonal_decompose
# Decompose data
decomp_results = seasonal_decompose(candy, period=12)

# Plot decomposed data
plt.rcParams["figure.figsize"] = (10,15)
decomp_results.plot()
plt.show()
```

### Original block 15

```text
# Subtract long rolling average over 5 steps
candy = candy - candy.rolling(5).mean()
# Drop NaN values
candy = candy.dropna()

# Identifying seasonal data using ACF

# Create figure
fig, ax = plt.subplots(1,1, figsize=(8,4))
# Plot ACF
plot_acf(candy.dropna(), ax=ax, lags=25, zero=False)
plt.show()
```

### Original block 16

```text
model = SARIMAX(df, order=(p,d,q), seasonal_order=(P,D,Q,S))
```

### Original block 17

```text
# Seasonal differencing
S = 12
candy_diff = candy.diff(S)
candy_diff.plot()
```

### Original block 18

```text
# one step differencing
candy_diff = candy.diff()
candy_diff = candy_diff.dropna()
candy_diff.plot()
```

### Original block 19

```text
# Run Dicky-Fuller test
result = adfuller(candy_diff)

# Print test statistic
print('The test stastics:', result[0])

# Print p-value
print("The p-value:",result[1])

```

### Original block 20

```text
# find the non-seasonal model prameters 

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Create figure
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(12,8))
 
# Plot the ACF of savings on ax1
plot_acf(candy_diff, lags=20, zero=False, ax=ax1)

# Plot the PACF of savings on ax2
plot_pacf(candy_diff, lags=20, zero=False, ax=ax2)

plt.show()
```

### Original block 21

```text
import statsmodels.api as sm
order_aic_bic =[]

# Loop over p values from 0-4
for p in range(5):
    # Loop over q values from 0-4
    for q in range(5):
      
        try:
            # create and fit ARMA(p,q) model
            model = sm.tsa.statespace.SARIMAX(candy_diff, order=(p, 1, q))
            results = model.fit()
            
            # Print order and results
            order_aic_bic.append((p, q, results.aic, results.bic))            
        except:
            print(p, q, None, None)
            
# Make DataFrame of model order and AIC/BIC scores
order_df = pd.DataFrame(order_aic_bic, columns=['p', 'q', 'aic','bic'])

# lets sort them by AIC and BIC

# Sort by AIC
print("Models sorted by AIC ")
print("\n")
print(order_df.sort_values('aic').reset_index(drop=True))

# Sort by BIC
print("Models sorted by BIC ")
print("\n")
print(order_df.sort_values('bic').reset_index(drop=True))
```

### Original block 22

```text
# Plotting seasonal ACF and PACF

# Create figure
fig, (ax1, ax2) = plt.subplots(2,1)

# Plot seasonal ACF
plot_acf(candy_diff, lags=[12,24,36,48,60,72, 84, 96 ], ax=ax1)

# Plot seasonal PACF
plot_pacf(candy_diff, lags=[12,24,36,48,60,72, 84, 96], ax=ax2)
plt.show()
```

### Original block 23

```text
# Fitting a SARIMA model
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Instantiate model
S = 12
D = 1
d = 1
P = 0
Q = 3
p = 4
q = 3
model = SARIMAX(candy, order=(p,d,q), seasonal_order=(P,D,Q,S))

# Fit model
results = model.fit()
```

### Original block 24

```text
# Searching over model orders
import pmdarima as pm

results = pm.auto_arima(candy)
print(results)
print(results.summary())
results.plot_diagnostics()
```

### Original block 25

```text
# Seasonal search parameters

results = pm.auto_arima(candy, # data
                        seasonal=True, # is the time series seasonal
                        m=12, # the seasonal period
                        D=1, # seasonal difference order
                        start_P=1, # initial guess for P
                        start_Q=1, # initial guess for Q
                        max_P=4, # max value of P to test
                        max_Q=4, # max value of Q to test
                        information_criterion='aic', # used to select best model
                        trace=True, # print results whilst training
                        error_action='ignore', # ignore orders that don't work
                        stepwise=True,
                       )
print(results)
print(results.summary())
results.plot_diagnostics()
```

### Original block 26

```text
#Saving model objects
# Import joblib
import joblib
# Select a filepath
filepath = 'model.pkl'
# Save model to filepath
joblib.dump(results, filepath)

# Load the model 
# Select a filepath
filepath ='model.pkl'
# Load model object from filepath
loaded_model = joblib.load(filepath)
```